# CS180 Project 3, Part B: Flow Matching from Scratch
Hovhannes Antablyan · CS 180 Fall 2026 · https://codegrox.github.io/cs180_proj3/

UNets trained on MNIST on a Colab Tesla T4. The trainer (`partb_train.py`) is written by the second cell and run as a
subprocess; results and per-epoch checkpoints are saved to `MyDrive/cs180_proj3_partb_final/`, and finished stages
are skipped on a rerun.

1. Required: single-step denoiser (σ = 0.5), out-of-distribution σ, pure-noise denoiser, time-conditioned flow matching,
   class-conditioned flow matching with CFG (γ = 5), and the no-scheduler comparison (constant Adam LR 2e-3).
2. CS280A bells & whistles: a better time-only UNet (wider, trained for 30 epochs, EMA), scored by a small MNIST classifier.
3. Own ideas: Euler step-count sweep (1–300 steps) and a noise-to-digits sampling animation.


In [12]:
import os, sys, json, time, subprocess
from pathlib import Path
import torch, torchvision
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__)
if not torch.cuda.is_available() and not os.environ.get('CS180_ALLOW_CPU'):
    raise RuntimeError('Enable a GPU: Runtime -> Change runtime type.')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive'); DATA_DIR = Path('/content/mnist_data')
elif Path('/kaggle/working').is_dir():
    ROOT = Path('/kaggle/working'); DATA_DIR = ROOT / 'mnist_data'
else:
    ROOT = Path.cwd(); DATA_DIR = Path(os.environ.get('CS180_MNIST', 'mnist_data'))
OUTPUT_DIR = Path(os.environ.get('CS180_RESULTS', ROOT / 'cs180_proj3_partb_final'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Results:', OUTPUT_DIR)


PyTorch 2.11.0+cu130 | torchvision 0.26.0+cu130
Device: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results: /content/drive/MyDrive/cs180_proj3_partb_final


In [13]:
trainer_source = '"""CS180 Fall 2026 Project 3 Part B — MNIST denoising / flow matching.\n\nReproducible local/Colab/Kaggle trainer. Outputs are *actual* figures only after\ntraining. Epoch checkpoint writes are atomic, and --resume restarts a run.\n\nNo network is used except torchvision\'s initial MNIST download. Not part of Part A.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import Tensor, nn\nimport torch.nn.functional as F\nimport torchvision\nfrom torchvision import transforms\nfrom torchvision.utils import make_grid, save_image\n\n# Torch defaults can oversubscribe CPUs in container smoke checks.\n\n\nclass Conv(nn.Sequential):\n    def __init__(self, c_in: int, c_out: int):\n        super().__init__(nn.Conv2d(c_in, c_out, 3, 1, 1), nn.BatchNorm2d(c_out), nn.GELU())\n\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, c_in: int, c_out: int):\n        super().__init__(Conv(c_in, c_out), Conv(c_out, c_out))\n\n\nclass DownBlock(nn.Sequential):\n    def __init__(self, c_in: int, c_out: int):\n        super().__init__(nn.Conv2d(c_in, c_out, 3, 2, 1),\n                         nn.BatchNorm2d(c_out), nn.GELU(), ConvBlock(c_out, c_out))\n\n\nclass UpBlock(nn.Sequential):\n    def __init__(self, c_in: int, c_out: int):\n        super().__init__(nn.ConvTranspose2d(c_in, c_out, 4, 2, 1),\n                         nn.BatchNorm2d(c_out), nn.GELU(), ConvBlock(c_out, c_out))\n\n\nclass FCBlock(nn.Sequential):\n    def __init__(self, c_in: int, c_out: int):\n        super().__init__(nn.Linear(c_in, c_out), nn.GELU(), nn.Linear(c_out, c_out))\n\n\nclass UNet(nn.Module):\n    """Follows the supplied 28→14→7→1→7→14→28 UNet schematic.\n\n    mode: \'denoise\' (unconditional), \'time\', or \'class\'.\n    class mode uses 10-way one-hot class vectors and zero vector for unconditional.\n    """\n    def __init__(self, D: int = 64, mode: str = \'denoise\'):\n        super().__init__()\n        if mode not in (\'denoise\', \'time\', \'class\'):\n            raise ValueError(mode)\n        self.D = D\n        self.mode = mode\n        self.conv0 = ConvBlock(1, D)\n        self.down1 = DownBlock(D, D)\n        self.down2 = DownBlock(D, 2 * D)\n        self.flatten = nn.Sequential(nn.AvgPool2d(7), nn.GELU())\n        self.unflatten = nn.Sequential(nn.ConvTranspose2d(2*D, 2*D, 7, 7),\n                                       nn.BatchNorm2d(2*D), nn.GELU())\n        self.up1 = UpBlock(4*D, D)\n        self.up2 = UpBlock(2*D, D)\n        self.conv_out = ConvBlock(2*D, D)\n        self.last = nn.Conv2d(D, 1, 3, 1, 1)\n        if mode != \'denoise\':\n            self.fc1_t = FCBlock(1, 2*D)\n            self.fc2_t = FCBlock(1, D)\n        if mode == \'class\':\n            self.fc1_c = FCBlock(10, 2*D)\n            self.fc2_c = FCBlock(10, D)\n\n    def forward(self, x: Tensor, t: Tensor | None = None, c: Tensor | None = None):\n        if x.ndim != 4 or x.shape[1:] != (1,28,28):\n            raise ValueError(f\'Expected (B,1,28,28); got {tuple(x.shape)}\')\n        b = x.shape[0]\n        x0 = self.conv0(x)                 # (B,D,28,28)\n        x1 = self.down1(x0)                # (B,D,14,14)\n        x2 = self.down2(x1)                # (B,2D,7,7)\n        z = self.unflatten(self.flatten(x2))  # (B,2D,7,7)\n        if self.mode != \'denoise\':\n            if t is None:\n                raise ValueError(\'Time required by time/class modes\')\n            t = t.reshape(b,1).to(x.device, x.dtype)\n            emb1 = self.fc1_t(t).reshape(b, 2*self.D,1,1)\n            emb2 = self.fc2_t(t).reshape(b, self.D,1,1)\n            if self.mode == \'class\':\n                if c is None:\n                    raise ValueError(\'One-hot class conditioning required; zero for unconditional\')\n                c = c.reshape(b,10).to(x.device, x.dtype)\n                z = self.fc1_c(c).reshape(b,2*self.D,1,1) * z + emb1\n            else:\n                z = z * emb1\n        z = self.up1(torch.cat([z,x2],dim=1))  # (B,D,14,14)\n        if self.mode != \'denoise\':\n            if self.mode == \'class\':\n                z = self.fc2_c(c).reshape(b,self.D,1,1) * z + emb2\n            else:\n                z = z * emb2\n        z = self.up2(torch.cat([z,x1],dim=1))\n        z = self.conv_out(torch.cat([z,x0],dim=1))\n        return self.last(z)\n\n\ndef one_hot(labels: Tensor) -> Tensor:\n    return F.one_hot(labels.long(), num_classes=10).float()\n\n\n@torch.no_grad()\ndef flow_sample(model: UNet, n: int, device: torch.device,\n                labels: Tensor | None = None, steps: int = 100, gamma: float = 5.0,\n                seed: int = 180) -> Tensor:\n    """Euler integrate 0→1. For class mode, CFG=u_uncond+gamma*(u_cond-u_uncond)."""\n    model.eval()\n    generator = torch.Generator(device=device).manual_seed(seed)\n    x = torch.randn((n,1,28,28),device=device,generator=generator)\n    if model.mode == \'class\':\n        if labels is None or len(labels) != n:\n            raise ValueError(\'Class sampling needs one label for each image\')\n        c = one_hot(labels.to(device))\n        c_null = torch.zeros_like(c)\n    for i in range(steps):\n        t = torch.full((n,1),i/steps,device=device)\n        if model.mode == \'class\':\n            u_u = model(x,t,c_null)\n            u_c = model(x,t,c)\n            v = u_u + gamma*(u_c-u_u)\n        else:\n            v = model(x,t)\n        x = x + (1.0/steps)*v\n    return x.clamp(0,1)\n\n\ndef figure_grid(images: Tensor, path: Path, nrow: int, title: str = \'\'):\n    path.parent.mkdir(parents=True,exist_ok=True)\n    save_image(images.detach().float().cpu().clamp(0,1),path, nrow=nrow,\n               padding=2, pad_value=0)\n\n\ndef plot_loss(history: list[dict], path: Path, title: str):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,ax=plt.subplots(figsize=(9,4))\n    steps=np.array([v[\'step\'] for v in history]); losses=np.array([v[\'loss\'] for v in history])\n    ax.plot(steps,losses,alpha=.55,label=\'minibatch MSE\')\n    if len(losses)>1:\n        k=min(len(losses),max(5,len(losses)//60))\n        # use valid convolution to avoid zero-padding bias\n        if len(losses)>=k:\n            ax.plot(steps[k-1:],np.convolve(losses,np.ones(k)/k,\'valid\'),label=f\'{k}-batch average\',linewidth=1.7)\n    ax.set(xlabel=\'Optimizer step\',ylabel=\'Mean squared error\',title=title)\n    ax.legend();ax.grid(alpha=.2);fig.tight_layout();fig.savefig(path,dpi=150);plt.close(fig)\n\n\ndef atomic_save(payload: dict, path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp=path.with_suffix(\'.tmp\')\n    torch.save(payload,temp)\n    os.replace(temp,path)\n\n\ndef snapshot_figure(stage: str, model: UNet, output: Path, epoch: int,\n                    test_images: Tensor, device: torch.device, steps: int):\n    was_training=model.training\n    model.eval()\n    with torch.no_grad():\n        if stage in (\'denoise\',\'pure\'):\n            gen=torch.Generator(device=device).manual_seed(991)\n            noise=torch.randn(test_images.shape,device=device,generator=gen)\n            if stage==\'denoise\':\n                noisy=test_images + .5*noise\n                denoised=model(noisy).clamp(0,1)\n                tiles=torch.cat([test_images.cpu(),noisy.clamp(0,1).cpu(),denoised.cpu()])\n                figure_grid(tiles, output/f\'{stage}_epoch{epoch}.png\', nrow=len(test_images))\n            else:\n                denoised=model(noise).clamp(0,1)\n                tiles=torch.cat([noise.clamp(0,1).cpu(),denoised.cpu()])\n                figure_grid(tiles, output/f\'{stage}_epoch{epoch}.png\', nrow=len(test_images))\n        else:\n            labels=torch.arange(10,device=device).repeat(4) if stage != \'time\' else None\n            sample=flow_sample(model,40,device,labels=labels,steps=steps,seed=180)\n            figure_grid(sample,output/f\'{stage}_epoch{epoch}.png\',nrow=10)\n    model.train(was_training)\n\n\n@torch.no_grad()\ndef out_of_distribution(model: UNet, x: Tensor, device: torch.device, output: Path):\n    """Compare same input digit + identical epsilon across noise scales (test set)."""\n    model.eval()\n    gen=torch.Generator(device=device).manual_seed(991)\n    epsilon=torch.randn(x.shape,device=device,generator=gen)\n    sigmas=[0.0,.2,.4,.5,.6,.8,1.0]\n    samples=[]\n    for sigma in sigmas:\n        noisy=x+sigma*epsilon\n        pred=model(noisy)\n        samples += [noisy.clamp(0,1).cpu(),pred.clamp(0,1).cpu()]\n    figure_grid(torch.cat(samples),output/\'denoise_ood.png\',nrow=x.shape[0])\n    (output/\'denoise_ood_labels.json\').write_text(json.dumps({\'rows\':[\'input_sigma_\'+str(s) if i%2==0 else \'denoised_sigma_\'+str(s) for s in sigmas for i in range(2)],\'sigma\':sigmas},indent=2))\n\n\ndef seed_all(seed: int):\n    random.seed(seed); np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\n\ndef load_mnist(data_dir: Path, batch_size: int, workers: int):\n    train = torchvision.datasets.MNIST(root=str(data_dir), train=True,\n                                        transform=transforms.ToTensor(), download=True)\n    test = torchvision.datasets.MNIST(root=str(data_dir), train=False,\n                                       transform=transforms.ToTensor(), download=True)\n    generator=torch.Generator().manual_seed(180)\n    indices=torch.randperm(len(train),generator=generator).tolist()\n    train=torch.utils.data.Subset(train,indices)\n    loader=torch.utils.data.DataLoader(train,batch_size=batch_size,shuffle=True,\n                                       num_workers=workers,pin_memory=torch.cuda.is_available(),\n                                       drop_last=True,persistent_workers=workers>0)\n    fixed=torch.stack([test[i][0] for i in (0,1,2,3,4,5,6,7)]).float()\n    return loader,fixed\n\n\ndef experiment(stage: str, args, device: torch.device, output: Path):\n    seed_all(args.seed)\n    name=stage\n    dest=output/name\n    dest.mkdir(parents=True,exist_ok=True)\n    flow=stage in (\'time\',\'class\',\'class_nosched\')\n    is_denoise=stage in (\'denoise\',\'pure\')\n    is_class=stage.startswith(\'class\')\n    epochs=args.flow_epochs if flow else args.denoise_epochs\n    D=args.denoise_D if is_denoise else args.flow_D\n    batch_size=args.denoise_batch if is_denoise else args.flow_batch\n    learning_rate=(args.no_scheduler_lr if stage==\'class_nosched\' else\n                   (args.denoise_lr if is_denoise else args.flow_lr))\n    model=UNet(D, \'class\' if is_class else (\'time\' if stage==\'time\' else \'denoise\')).to(device)\n    opt=torch.optim.Adam(model.parameters(),lr=learning_rate)\n    sched=(torch.optim.lr_scheduler.ExponentialLR(opt, gamma=.1**(1/epochs))\n           if flow and stage != \'class_nosched\' else None)\n    loader, fixed = load_mnist(args.data_dir, batch_size, args.workers)\n    fixed=fixed.to(device)\n    print(f\'[{stage}] device={device}, width={D}, examples={len(loader.dataset)}, \'\n          f\'batches={len(loader)}, lr={learning_rate}, epochs={epochs}\',flush=True)\n    start_epoch=1; global_step=0; history=[]\n    checkpoint=dest/\'checkpoint_latest.pt\'\n    if args.resume and checkpoint.exists():\n        # Own checkpoints only; weights_only=False required for optimizer RNG states.\n        ckpt=torch.load(checkpoint,map_location=device,weights_only=False)\n        assert ckpt[\'stage\']==stage and ckpt[\'D\']==D\n        model.load_state_dict(ckpt[\'model\']);opt.load_state_dict(ckpt[\'optimizer\'])\n        if sched is not None and ckpt[\'scheduler\'] is not None:sched.load_state_dict(ckpt[\'scheduler\'])\n        start_epoch=ckpt[\'epoch\']+1;global_step=ckpt[\'global_step\'];history=ckpt[\'history\']\n        torch.set_rng_state(ckpt[\'torch_rng\'].cpu())\n        if device.type==\'cuda\' and ckpt[\'cuda_rng\'] is not None:\n            torch.cuda.set_rng_state_all(ckpt[\'cuda_rng\'])\n        print(f\'[{stage}] Resuming at epoch {start_epoch}\',flush=True)\n    (dest/\'config.json\').write_text(json.dumps({\n        \'stage\':stage,\'width_D\':D,\'epochs\':epochs,\'batch_size\':batch_size,\n        \'optimizer\':\'Adam\',\'learning_rate\':learning_rate,\'scheduler\':(f\'ExponentialLR gamma={.1**(1/epochs)}\' if sched else \'none\'),\n        \'fixed_lr_compensation\': (\'constant Adam lr=%.5g\'%learning_rate if stage==\'class_nosched\' else None),\n        \'steps\':args.steps,\'seed\':args.seed,\'training_dataset\':\'torchvision MNIST train\',\n        \'training_examples\':len(loader.dataset)},indent=2))\n    if start_epoch==1 and stage==\'denoise\':\n        epsilon=torch.randn_like(fixed[:1])\n        examples=torch.cat([(fixed[:1]+s*epsilon).clamp(0,1) for s in [0,.2,.4,.5,.6,.8,1]])\n        figure_grid(examples,dest/\'noise_levels.png\',nrow=7)\n    selected={1,5,10}\n    if not flow:selected={1,epochs}\n    t0=time.time()\n    for epoch in range(start_epoch,epochs+1):\n        model.train()\n        for batch,(clean,labels) in enumerate(loader):\n            clean=clean.to(device,non_blocking=True);labels=labels.to(device,non_blocking=True)\n            eps=torch.randn_like(clean)\n            if is_denoise:\n                input_t = clean + .5*eps if stage==\'denoise\' else eps\n                prediction=model(input_t)\n                target=clean\n            else:\n                t=torch.rand((len(clean),1,1,1),device=device)\n                x_t=(1-t)*eps + t*clean\n                target=clean-eps\n                if is_class:\n                    c=one_hot(labels)\n                    drop=(torch.rand((len(clean),1),device=device)<.1)\n                    c=c.masked_fill(drop,0)\n                    prediction=model(x_t,t.reshape(-1,1),c)\n                else:prediction=model(x_t,t.reshape(-1,1))\n            loss=F.mse_loss(prediction,target)\n            opt.zero_grad(set_to_none=True);loss.backward();opt.step()\n            global_step+=1\n            history.append({\'step\':global_step,\'epoch\':epoch,\'loss\':float(loss.detach().item()),\n                            \'lr\':opt.param_groups[0][\'lr\']})\n            if args.max_batches and batch+1 >= args.max_batches: break  # smoke/diagnostic only\n            if global_step % args.log_every==0:\n                print(f\'[{stage}] epoch={epoch}/{epochs} batch={batch+1}/{len(loader)} \'\n                      f\'loss={history[-1]["loss"]:.5f}\',flush=True)\n        if sched is not None:sched.step()\n        if epoch in selected:\n            snapshot_figure(stage,model,dest,epoch,fixed,device,args.steps)\n            if stage==\'denoise\' and epoch==epochs:\n                out_of_distribution(model,fixed,device,dest)\n        plot_loss(history,dest/\'training_loss.png\',title=f\'{stage} · MNIST training loss\')\n        (dest/\'loss_history.csv\').write_text(\'step,epoch,loss,lr\\n\'+\'\'.join(\n            f\'{v["step"]},{v["epoch"]},{v["loss"]:.8g},{v["lr"]:.8g}\\n\' for v in history))\n        atomic_save({\'stage\':stage,\'D\':D,\'epoch\':epoch,\'global_step\':global_step,\n                     \'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\n                     \'scheduler\':sched.state_dict() if sched is not None else None,\n                     \'history\':history,\'torch_rng\':torch.get_rng_state(),\n                     \'cuda_rng\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else None},checkpoint)\n        print(f\'[{stage}] epoch {epoch} done | checkpoint saved | elapsed={time.time()-t0:.0f}s\',flush=True)\n    if start_epoch > epochs:\n        print(f\'[{stage}] Completed previously; no retraining needed.\',flush=True)\n\n\ndef smoke_test():\n    print(\'Running CPU smoke test (random synthetic tensors, not MNIST results)...\')\n    torch.set_num_threads(2)\n    for mode in (\'denoise\',\'time\',\'class\'):\n        net=UNet(4,mode)\n        x=torch.randn(2,1,28,28)\n        t=torch.rand(2,1); c=one_hot(torch.tensor([0,9]))\n        out=net(x,t if mode!=\'denoise\' else None,c if mode==\'class\' else None)\n        assert out.shape==x.shape and torch.isfinite(out).all()\n        out.square().mean().backward()\n        assert all(p.grad is not None for p in net.parameters())\n        if mode!=\'denoise\':\n            sampled=flow_sample(net,2,torch.device(\'cpu\'),labels=torch.tensor([0,9])\n                                if mode==\'class\' else None, steps=2)\n            assert sampled.shape==x.shape\n        print(f\' PASS {mode}: output/backward/sampler shapes\',flush=True)\n    print(\'All synthetic checks passed. This does not count as training.\')\n\n\ndef main():\n    ap=argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--experiments\',default=\'denoise,pure,time,class,class_nosched\',\n                    help=\'Comma-separated names, in execution order\')\n    ap.add_argument(\'--data-dir\',type=Path,default=Path(\'data/mnist\'))\n    ap.add_argument(\'--output-dir\',type=Path,default=Path(\'out/partb\'))\n    ap.add_argument(\'--device\',choices=[\'auto\',\'cuda\',\'cpu\'],default=\'auto\')\n    ap.add_argument(\'--resume\',action=\'store_true\',help=\'Continue from latest epoch checkpoints\')\n    ap.add_argument(\'--smoke\',action=\'store_true\',help=\'Only validate CPU architecture with synthetic tensors\')\n    ap.add_argument(\'--seed\',type=int,default=180)\n    ap.add_argument(\'--denoise-D\',type=int,default=128)\n    ap.add_argument(\'--flow-D\',type=int,default=64)\n    ap.add_argument(\'--denoise-batch\',type=int,default=256)\n    ap.add_argument(\'--flow-batch\',type=int,default=64)\n    ap.add_argument(\'--denoise-epochs\',type=int,default=5)\n    ap.add_argument(\'--flow-epochs\',type=int,default=10)\n    ap.add_argument(\'--denoise-lr\',type=float,default=1e-4)\n    ap.add_argument(\'--flow-lr\',type=float,default=1e-2)\n    ap.add_argument(\'--no-scheduler-lr\',type=float,default=0.002,\n                    help=\'Constant LR for scheduler ablation (run all 10 epochs)\')\n    ap.add_argument(\'--steps\',type=int,default=100)\n    ap.add_argument(\'--workers\',type=int,default=2)\n    ap.add_argument(\'--log-every\',type=int,default=30)\n    ap.add_argument(\'--max-batches\',type=int,default=0,help=\'Diagnostic only, invalidates full assignment results\')\n    args=ap.parse_args()\n    if args.smoke:return smoke_test()\n    if args.device==\'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA was requested but no GPU is allocated. Enable GPU in Colab/Kaggle.\')\n    device=torch.device((\'cuda\' if torch.cuda.is_available() else \'cpu\') if args.device==\'auto\' else args.device)\n    if device.type==\'cpu\':\n        print(\'WARNING: CPU mode can take hours; GPU strongly recommended.\',flush=True)\n    for stage in args.experiments.split(\',\'):\n        if stage not in (\'denoise\',\'pure\',\'time\',\'class\',\'class_nosched\'):raise ValueError(stage)\n        experiment(stage,args,device,args.output_dir)\n    print(\'All requested experiments finished. Figures in\',args.output_dir,flush=True)\n\nif __name__==\'__main__\':main()\n'
Path('partb_train.py').write_text(trainer_source)
if '' not in sys.path:
    sys.path.insert(0, '')
import importlib, partb_train
importlib.reload(partb_train)
from partb_train import UNet, flow_sample, load_mnist, figure_grid, plot_loss, seed_all, atomic_save
subprocess.run([sys.executable, 'partb_train.py', '--smoke'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'partb_train.py', '--smoke'], returncode=0)

## Required experiments


In [14]:
# Required experiments (Sections 1.2.1-2.6). Each epoch is checkpointed; rerunning resumes.
# Euler steps follow the starter notebook: 50 for the time-conditioned model, 300 for class-conditioned (CFG gamma = 5).
# Stages whose final checkpoint already exists are skipped, and the trainer's own log is shown here, errors included.
EXTRA_ARGS = os.environ.get('CS180_TRAIN_ARGS', '').split()          # optional overrides for quick tests; empty here
TARGET_EPOCHS = {'denoise': 5, 'pure': 5, 'time': 10, 'class': 10, 'class_nosched': 10}
if '--flow-epochs' in EXTRA_ARGS:
    TARGET_EPOCHS.update({k: int(EXTRA_ARGS[EXTRA_ARGS.index('--flow-epochs') + 1]) for k in ('time', 'class', 'class_nosched')})
if '--denoise-epochs' in EXTRA_ARGS:
    TARGET_EPOCHS.update({k: int(EXTRA_ARGS[EXTRA_ARGS.index('--denoise-epochs') + 1]) for k in ('denoise', 'pure')})

def finished(stage):
    ck = OUTPUT_DIR / stage / 'checkpoint_latest.pt'
    if not ck.is_file():
        return False
    try:
        return torch.load(ck, map_location='cpu', weights_only=False)['epoch'] >= TARGET_EPOCHS[stage]
    except Exception as e:
        print(f'[{stage}] checkpoint unreadable ({e}); it will be retrained from its last good state')
        return False

def stages(names, steps):
    todo = [n for n in names.split(',') if not finished(n)]
    for n in names.split(','):
        if n not in todo:
            print(f'[{n}] already finished: skipped')
    if not todo:
        return
    cmd = [sys.executable, '-u', 'partb_train.py', '--experiments', ','.join(todo), '--resume', '--steps', str(steps),
           '--output-dir', str(OUTPUT_DIR), '--data-dir', str(DATA_DIR), '--log-every', '200'] + EXTRA_ARGS
    print('Running:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:                                        # stream the trainer's log into this cell
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'trainer exited with code {proc.returncode}; see the log above')

t0 = time.time()
stages('denoise,pure,time', 50)
stages('class,class_nosched', 300)
print(f'required experiments done ({(time.time() - t0) / 60:.1f} min in this session)')


[denoise] already finished: skipped
[pure] already finished: skipped
[time] already finished: skipped
[class] already finished: skipped
[class_nosched] already finished: skipped
required experiments done (0.0 min in this session)


In [15]:
# Section 1.2 figure with ten test digits (the trainer's own figure shows one digit): rows sigma = 0 ... 1.
import torchvision.transforms as T
test = torchvision.datasets.MNIST(str(DATA_DIR), train=False, transform=T.ToTensor(), download=True)
x10 = torch.stack([test[i][0] for i in range(10)])
e10 = torch.randn(x10.shape, generator=torch.Generator().manual_seed(180))
figure_grid(torch.cat([(x10 + s * e10).clamp(0, 1) for s in (0, .2, .4, .5, .6, .8, 1.)]),
            OUTPUT_DIR / 'denoise' / 'noise_levels_10.png', nrow=10)
print('saved noise_levels_10.png')


saved noise_levels_10.png


## Bells & Whistles: a better time-conditioned UNet (required for CS280A)


In [16]:
# CS280A bells & whistles: a better time-conditioned UNet. Wider (D = 128), longer (15 epochs, LR 5e-3 -> 5e-4),
# sampled from an EMA of the weights (0.999). Same 40 starting noises and 50 Euler steps as the baseline time grid.
OUT = OUTPUT_DIR / 'time_plus'
OUT.mkdir(exist_ok=True)
BASELINE = OUTPUT_DIR / 'time' / 'checkpoint_latest.pt'
import torch.nn as nn, torch.nn.functional as F
D, EPOCHS, LR, FINAL_LR_FACTOR, EMA_DECAY, BATCH, STEPS = 128, 15, 5e-3, 0.1, 0.999, 64, 50
SAMPLE_EPOCHS = {1, 5, 10, 15}
MAX_BATCHES = int(os.environ.get('CS180_MAX_BATCHES', 0))      # 0 = full epochs
EPOCHS = int(os.environ.get('CS180_EPOCHS', EPOCHS))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed_all(180)
model = UNet(D, 'time').to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=FINAL_LR_FACTOR ** (1 / EPOCHS))
ema = torch.optim.swa_utils.AveragedModel(model, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(EMA_DECAY),
                                          use_buffers=True)
loader, _ = load_mnist(DATA_DIR, BATCH, 2)
ck_path = OUT / 'checkpoint_latest.pt'
start, step, history, epoch_seconds = 1, 0, [], []
if ck_path.is_file():
    ck = torch.load(ck_path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer']); sched.load_state_dict(ck['scheduler'])
    ema.load_state_dict(ck['ema']); torch.set_rng_state(ck['torch_rng'].cpu())
    if ck['cuda_rng'] is not None: torch.cuda.set_rng_state_all([r.cpu() for r in ck['cuda_rng']])
    start, step, history, epoch_seconds = ck['epoch'] + 1, ck['step'], ck['history'], ck['epoch_seconds']
    print('resuming at epoch', start)
print('parameters:', sum(p.numel() for p in model.parameters()), '| batches/epoch:', len(loader))
for epoch in range(start, EPOCHS + 1):
    model.train(); t0 = time.time()
    for b, (clean, _) in enumerate(loader):
        clean = clean.to(device, non_blocking=True)
        eps = torch.randn_like(clean)
        t = torch.rand((len(clean), 1, 1, 1), device=device)
        loss = F.mse_loss(model((1 - t) * eps + t * clean, t.reshape(-1, 1)), clean - eps)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); ema.update_parameters(model)
        step += 1
        history.append({'step': step, 'epoch': epoch, 'loss': float(loss.detach()), 'lr': opt.param_groups[0]['lr']})
        if MAX_BATCHES and b + 1 >= MAX_BATCHES: break
    sched.step()
    epoch_seconds.append(time.time() - t0)
    if epoch in SAMPLE_EPOCHS:
        x = flow_sample(ema.module, 40, device, steps=STEPS, seed=180)
        torch.save(x.cpu(), OUT / f'time_plus_epoch{epoch}.pt')
        figure_grid(x, OUT / f'time_plus_epoch{epoch}.png', nrow=10)
    plot_loss(history, OUT / 'training_loss.png', 'time-conditioned UNet, D=128, 15 epochs')
    (OUT / 'loss_history.csv').write_text('step,epoch,loss,lr\n' + ''.join(
        f"{h['step']},{h['epoch']},{h['loss']:.8g},{h['lr']:.8g}\n" for h in history))
    atomic_save({'epoch': epoch, 'step': step, 'model': model.state_dict(), 'optimizer': opt.state_dict(),
                 'scheduler': sched.state_dict(), 'ema': ema.state_dict(), 'history': history,
                 'epoch_seconds': epoch_seconds, 'torch_rng': torch.get_rng_state(),
                 'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else None}, ck_path)
    print(f'epoch {epoch}/{EPOCHS}: mean loss {sum(h["loss"] for h in history if h["epoch"] == epoch) / max(1, sum(h["epoch"] == epoch for h in history)):.4f}, '
          f'{epoch_seconds[-1]:.0f}s, checkpoint saved', flush=True)


resuming at epoch 16
parameters: 7974145 | batches/epoch: 937


In [17]:
# Legibility / diversity of 1,000 samples per model, judged by a small classifier trained on real MNIST.
import torchvision
from torchvision import transforms
class Judge(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(),
                                 nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64 * 49, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x): return self.net(x)
seed_all(7)
judge = Judge().to(device); jopt = torch.optim.Adam(judge.parameters(), 1e-3)
jl, _ = load_mnist(DATA_DIR, 128, 2)
for e in range(2):
    for b, (x, y) in enumerate(jl):
        l = F.cross_entropy(judge(x.to(device)), y.to(device)); jopt.zero_grad(); l.backward(); jopt.step()
        if MAX_BATCHES and b + 1 >= MAX_BATCHES: break
judge.eval()
test = torchvision.datasets.MNIST(str(DATA_DIR), train=False, transform=transforms.ToTensor(), download=True)
with torch.no_grad():
    xs = torch.stack([test[i][0] for i in range(len(test))]).to(device); ys = torch.tensor([test[i][1] for i in range(len(test))])
    judge_acc = float((judge(xs).argmax(1).cpu() == ys).float().mean())
print('judge accuracy on real test digits:', round(judge_acc, 4))

def score(net, n=1000):
    x = torch.cat([flow_sample(net, 250, device, steps=STEPS, seed=1000 + k) for k in range(n // 250)])
    with torch.no_grad():
        p = F.softmax(judge(x), 1)
    conf, cls = p.max(1)
    hist = torch.bincount(cls.cpu(), minlength=10).float() / len(cls)
    entropy = float(-(hist[hist > 0] * hist[hist > 0].log()).sum() / torch.log(torch.tensor(10.)))
    return {'mean_confidence': float(conf.mean()), 'frac_confident_0.9': float((conf > 0.9).float().mean()),
            'digit_histogram': [round(float(h), 3) for h in hist], 'normalized_entropy': entropy, 'n': len(cls)}

results = {'judge_test_accuracy': judge_acc, 'config': {'D': D, 'epochs': EPOCHS, 'lr': LR, 'final_lr': LR * FINAL_LR_FACTOR,
           'ema_decay': EMA_DECAY, 'batch': BATCH, 'euler_steps': STEPS, 'params': sum(p.numel() for p in model.parameters())},
           'epoch_seconds': [round(s, 1) for s in epoch_seconds]}
results['improved'] = score(ema.module)
if BASELINE.is_file():
    base = UNet(64, 'time').to(device)
    ck = torch.load(BASELINE, map_location=device, weights_only=False)
    assert ck['stage'] == 'time' and ck['D'] == 64
    base.load_state_dict(ck['model'])
    results['baseline'] = score(base)
    xb = flow_sample(base, 40, device, steps=STEPS, seed=180)
    torch.save(xb.cpu(), OUT / 'time_baseline_epoch10.pt')
(OUT / 'metrics.json').write_text(json.dumps(results, indent=2))
print(json.dumps({k: v for k, v in results.items() if k in ('improved', 'baseline')}, indent=1))


judge accuracy on real test digits: 0.9859
{
 "improved": {
  "mean_confidence": 0.9202954173088074,
  "frac_confident_0.9": 0.7660000324249268,
  "digit_histogram": [
   0.078,
   0.117,
   0.084,
   0.114,
   0.107,
   0.089,
   0.097,
   0.135,
   0.095,
   0.084
  ],
  "normalized_entropy": 0.9938374757766724,
  "n": 1000
 },
 "baseline": {
  "mean_confidence": 0.9073451161384583,
  "frac_confident_0.9": 0.7300000190734863,
  "digit_histogram": [
   0.05,
   0.084,
   0.105,
   0.116,
   0.08,
   0.112,
   0.072,
   0.153,
   0.122,
   0.106
  ],
  "normalized_entropy": 0.9827463626861572,
  "n": 1000
 }
}


## Own ideas


In [18]:
# Own idea: Euler step-count sweep. How many steps do the trained flows need?
# Class-conditioned model from the main run (gamma = 5) and the improved time-only model, same starting noise for every n.
STEP_SWEEP = [1, 2, 5, 10, 25, 50, 100, 300]
sweep = {'steps': STEP_SWEEP}
eval_labels = torch.arange(10, device=device).repeat_interleave(50)
grid_labels = torch.arange(10, device=device).repeat(4)
CLASS_CKPT = OUTPUT_DIR / 'class' / 'checkpoint_latest.pt'
if CLASS_CKPT.is_file():
    cnet = UNet(64, 'class').to(device)
    ck = torch.load(CLASS_CKPT, map_location=device, weights_only=False)
    assert ck['stage'] == 'class' and ck['D'] == 64
    cnet.load_state_dict(ck['model'])
    sweep['class_accuracy'] = {}
    for n in STEP_SWEEP:
        g = flow_sample(cnet, 40, device, labels=grid_labels, steps=n, seed=180)
        torch.save(g.cpu(), OUT / f'steps_class_{n}.pt'); figure_grid(g, OUT / f'steps_class_{n}.png', nrow=10)
        xe = flow_sample(cnet, 500, device, labels=eval_labels, steps=n, seed=2000)
        with torch.no_grad():
            sweep['class_accuracy'][n] = float((judge(xe).argmax(1) == eval_labels).float().mean())
        print(f'class model, {n} steps: {100 * sweep["class_accuracy"][n]:.1f}% read as the requested digit', flush=True)
else:
    print('Main-run class checkpoint not found (expected on Colab at', CLASS_CKPT, '); class sweep skipped.')
sweep['time_plus_confidence'] = {}
for n in STEP_SWEEP:
    g = flow_sample(ema.module, 40, device, steps=n, seed=180)
    torch.save(g.cpu(), OUT / f'steps_time_plus_{n}.pt'); figure_grid(g, OUT / f'steps_time_plus_{n}.png', nrow=10)
    xe = flow_sample(ema.module, 500, device, steps=n, seed=2000)
    with torch.no_grad():
        conf = F.softmax(judge(xe), 1).max(1).values
    sweep['time_plus_confidence'][n] = {'mean': float(conf.mean()), 'frac_0.9': float((conf > 0.9).float().mean())}
results['step_sweep'] = sweep
(OUT / 'metrics.json').write_text(json.dumps(results, indent=2))
print(json.dumps(sweep, indent=1))


class model, 1 steps: 100.0% read as the requested digit
class model, 2 steps: 100.0% read as the requested digit
class model, 5 steps: 100.0% read as the requested digit
class model, 10 steps: 100.0% read as the requested digit
class model, 25 steps: 100.0% read as the requested digit
class model, 50 steps: 100.0% read as the requested digit
class model, 100 steps: 100.0% read as the requested digit
class model, 300 steps: 100.0% read as the requested digit
{
 "steps": [
  1,
  2,
  5,
  10,
  25,
  50,
  100,
  300
 ],
 "class_accuracy": {
  "1": 1.0,
  "2": 1.0,
  "5": 1.0,
  "10": 1.0,
  "25": 1.0,
  "50": 1.0,
  "100": 1.0,
  "300": 1.0
 },
 "time_plus_confidence": {
  "1": {
   "mean": 0.24559715390205383,
   "frac_0.9": 0.0
  },
  "2": {
   "mean": 0.7153339982032776,
   "frac_0.9": 0.328000009059906
  },
  "5": {
   "mean": 0.8689696192741394,
   "frac_0.9": 0.6300000548362732
  },
  "10": {
   "mean": 0.8980700373649597,
   "frac_0.9": 0.706000030040741
  },
  "25": {
   "mean

In [19]:
# Own idea: animation of the epoch-10 class-conditioned sampler, noise -> digits (300 Euler steps, gamma = 5).
from PIL import Image as _PIL
from torchvision.utils import make_grid
labels = torch.arange(10, device=device).repeat(4)
c = F.one_hot(labels, 10).float(); c0 = torch.zeros_like(c)
x = torch.randn((40, 1, 28, 28), device=device, generator=torch.Generator(device=device).manual_seed(180))
frames, n = [], 300
with torch.no_grad():
    cnet.eval()
    for i in range(n):
        t = torch.full((40, 1), i / n, device=device)
        u0, u1 = cnet(x, t, c0), cnet(x, t, c)
        x = x + (u0 + 5.0 * (u1 - u0)) / n
        if i % 10 == 9 or i == 0:
            g = make_grid(x.clamp(0, 1).cpu(), nrow=10, padding=2)[0]
            frames.append(_PIL.fromarray((g * 255).byte().numpy()).resize((g.shape[1] * 3, g.shape[0] * 3), _PIL.NEAREST))
frames[0].save(OUTPUT_DIR / 'class' / 'class_sampling.gif', save_all=True, append_images=frames[1:] + [frames[-1]] * 12,
               duration=70, loop=0)
print('saved class_sampling.gif with', len(frames), 'frames')


saved class_sampling.gif with 31 frames


In [20]:
# CS280A, continued: the same D = 128 model and its EMA trained for 15 more epochs (30 in total), with the
# learning rate decaying again from 5e-4 to 5e-5, then scored exactly like the baseline and the 15-epoch model.
EXTRA_EPOCHS = int(os.environ.get('CS180_EXTRA_EPOCHS', 15))
OUT2 = OUTPUT_DIR / 'time_plus30'
OUT2.mkdir(exist_ok=True)
ck2 = OUT2 / 'checkpoint_latest.pt'
base_ck = torch.load(OUT / 'checkpoint_latest.pt', map_location=device, weights_only=False)
model2 = UNet(D, 'time').to(device)
ema2 = torch.optim.swa_utils.AveragedModel(model2, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(EMA_DECAY),
                                           use_buffers=True)
opt2 = torch.optim.Adam(model2.parameters(), lr=5e-4)
sched2 = torch.optim.lr_scheduler.ExponentialLR(opt2, gamma=0.1 ** (1 / EXTRA_EPOCHS))
if ck2.is_file():
    ck = torch.load(ck2, map_location=device, weights_only=False)
    model2.load_state_dict(ck['model']); opt2.load_state_dict(ck['optimizer']); sched2.load_state_dict(ck['scheduler'])
    ema2.load_state_dict(ck['ema']); torch.set_rng_state(ck['torch_rng'].cpu())
    if ck['cuda_rng'] is not None: torch.cuda.set_rng_state_all([r.cpu() for r in ck['cuda_rng']])
    start2, step2, hist2, secs2 = ck['epoch'] + 1, ck['step'], ck['history'], ck['epoch_seconds']
    print('resuming at epoch', start2)
else:                                   # start from the finished 15-epoch model and its EMA
    model2.load_state_dict(base_ck['model']); ema2.load_state_dict(base_ck['ema'])
    start2, step2, hist2, secs2 = base_ck['epoch'] + 1, base_ck['step'], list(base_ck['history']), []
last = base_ck['epoch'] + EXTRA_EPOCHS
for epoch in range(start2, last + 1):
    model2.train(); t0 = time.time()
    for b, (clean, _) in enumerate(loader):
        clean = clean.to(device, non_blocking=True)
        eps = torch.randn_like(clean)
        t = torch.rand((len(clean), 1, 1, 1), device=device)
        loss = F.mse_loss(model2((1 - t) * eps + t * clean, t.reshape(-1, 1)), clean - eps)
        opt2.zero_grad(set_to_none=True); loss.backward(); opt2.step(); ema2.update_parameters(model2)
        step2 += 1
        hist2.append({'step': step2, 'epoch': epoch, 'loss': float(loss.detach()), 'lr': opt2.param_groups[0]['lr']})
        if MAX_BATCHES and b + 1 >= MAX_BATCHES: break
    sched2.step()
    secs2.append(time.time() - t0)
    if epoch in (20, 25, last):
        x = flow_sample(ema2.module, 40, device, steps=STEPS, seed=180)
        torch.save(x.cpu(), OUT2 / f'time_plus_epoch{epoch}.pt')
        figure_grid(x, OUT2 / f'time_plus_epoch{epoch}.png', nrow=10)
    plot_loss(hist2, OUT2 / 'training_loss.png', f'time-conditioned UNet, D=128, {last} epochs')
    (OUT2 / 'loss_history.csv').write_text('step,epoch,loss,lr\n' + ''.join(
        f"{h['step']},{h['epoch']},{h['loss']:.8g},{h['lr']:.8g}\n" for h in hist2))
    atomic_save({'epoch': epoch, 'step': step2, 'model': model2.state_dict(), 'optimizer': opt2.state_dict(),
                 'scheduler': sched2.state_dict(), 'ema': ema2.state_dict(), 'history': hist2, 'epoch_seconds': secs2,
                 'torch_rng': torch.get_rng_state(),
                 'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else None}, ck2)
    n_ep = sum(h['epoch'] == epoch for h in hist2)
    print(f'epoch {epoch}/{last}: mean loss {sum(h["loss"] for h in hist2 if h["epoch"] == epoch) / max(1, n_ep):.4f}, '
          f'{secs2[-1]:.0f}s, checkpoint saved', flush=True)
results['improved_30'] = score(ema2.module)
results['improved_30']['epochs'] = last
(OUT / 'metrics.json').write_text(json.dumps(results, indent=2))
for k in ('baseline', 'improved', 'improved_30'):
    if k in results:
        r = results[k]
        print(f'{k:12s} confident >0.9: {100 * r["frac_confident_0.9"]:.1f}% | mean confidence {r["mean_confidence"]:.3f} '
              f'| digit-balance entropy {r["normalized_entropy"]:.3f}')
import shutil, tempfile
tmp = Path(tempfile.mkdtemp()) / 'partb_final'
shutil.copytree(OUTPUT_DIR, tmp, ignore=shutil.ignore_patterns('checkpoint_latest.pt', '*.tmp', 'MNIST'))
z = shutil.make_archive(str(OUTPUT_DIR.parent / 'cs180_proj3_partb_final'), 'zip', tmp)
print('Re-zipped:', z)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(z)

epoch 16/30: mean loss 0.0879, 107s, checkpoint saved
epoch 17/30: mean loss 0.0872, 107s, checkpoint saved
epoch 18/30: mean loss 0.0871, 107s, checkpoint saved
epoch 19/30: mean loss 0.0869, 107s, checkpoint saved
epoch 20/30: mean loss 0.0866, 107s, checkpoint saved
epoch 21/30: mean loss 0.0865, 107s, checkpoint saved
epoch 22/30: mean loss 0.0861, 107s, checkpoint saved
epoch 23/30: mean loss 0.0858, 107s, checkpoint saved
epoch 24/30: mean loss 0.0857, 107s, checkpoint saved
epoch 25/30: mean loss 0.0856, 107s, checkpoint saved
epoch 26/30: mean loss 0.0855, 107s, checkpoint saved
epoch 27/30: mean loss 0.0856, 107s, checkpoint saved
epoch 28/30: mean loss 0.0857, 107s, checkpoint saved
epoch 29/30: mean loss 0.0852, 107s, checkpoint saved
epoch 30/30: mean loss 0.0852, 107s, checkpoint saved
baseline     confident >0.9: 73.0% | mean confidence 0.907 | digit-balance entropy 0.983
improved     confident >0.9: 76.6% | mean confidence 0.920 | digit-balance entropy 0.994
improved_30 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Save


In [21]:
import shutil, tempfile
tmp = Path(tempfile.mkdtemp()) / 'partb_final'
shutil.copytree(OUTPUT_DIR, tmp, ignore=shutil.ignore_patterns('checkpoint_latest.pt', '*.tmp', 'MNIST'))
z = shutil.make_archive(str(OUTPUT_DIR.parent / 'cs180_proj3_partb_final'), 'zip', tmp)
print('Download:', z)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(z)


Download: /content/drive/MyDrive/cs180_proj3_partb_final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>